In [6]:
import os
import re
import time
import torch
from PIL import Image
from diffusers import StableDiffusionPipeline

# ----------------------------
# Configuration
# ----------------------------

# Set your text prompt here
prompt = "a dog with glasses"

# Image output directory
output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------
# Device selection: MPS (Apple Silicon), else CPU
# ----------------------------

if torch.backends.mps.is_available():
    device = "mps"  # macOS with Apple Silicon
else:
    device = "cpu"  # CPU fallback for all platforms

print(f"Using device: {device}")

# ----------------------------
# Load Stable Diffusion v1.4
# ----------------------------

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float32,  # float32 for MPS or CPU (no float16 on Mac)
    safety_checker=None         # Disable NSFW filter; set to True if needed
)
pipe.to(device)

# ----------------------------
# Sanitize prompt for filename (safe across platforms)
# ----------------------------

def sanitize_filename(text):
    return re.sub(r"[^\w\-_\. ]", "_", text).strip().replace(" ", "_")

base_filename = sanitize_filename(prompt)

# ----------------------------
# Ensure unique filename (timestamp + counter if needed)
# ----------------------------

def generate_unique_filename(base_name, folder, ext="png"):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    i = 1
    while True:
        filename = f"{base_name}_{timestamp}_{i}.{ext}"
        filepath = os.path.join(folder, filename)
        if not os.path.exists(filepath):
            return filepath
        i += 1

filepath = generate_unique_filename(base_filename, output_dir)

# ----------------------------
# Generate the image
# ----------------------------

# Use autocast for slight performance gain on MPS
with torch.autocast("cpu" if device == "cpu" else "mps"):
    result = pipe(prompt)
    image = result.images[0]

# ----------------------------
# Save the image
# ----------------------------

image.save(filepath)
print(f"Image saved to: {filepath}")

# Display in notebook
image.show()

Using device: mps


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 18.96it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
100%|██████████| 50/50 [00:33<00:00,  1.51it/s]


Image saved to: generated_images/a_dog_with_glasses_20250416-011327_1.png
